In [14]:
### import all the necessary libraries 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import warnings 
warnings.filterwarnings("ignore") 

from sklearn.model_selection import train_test_split 



In [15]:
## load the dataset 
data = pd.read_csv("Churn_Modelling.csv") 
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


#### Here first three features are not required.. 
* and on the basis of other (past data) features we are going to predict customer will churn bank or not

In [16]:
## preprocess the data  
# drop irrelevent feartures 
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1) 
data.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [17]:
## sepearate the dependent and independent features 
X = data.drop("Exited",axis=1) 
y = data["Exited"]

In [18]:
X.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,Female,42,2,0.00,1,1,1,101348.88
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58
2,502,France,Female,42,8,159660.80,3,1,0,113931.57
3,699,France,Female,39,1,0.00,2,0,0,93826.63
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10


In [19]:
y.head()

0    1
1    0
2    1
3    0
4    0
Name: Exited, dtype: int64

### here in X fearures there are two categoical features lets convert into numerical features 
* but first we will identify the categorical and numerical features seperately

In [20]:
categorical_features = X.select_dtypes(include='object').columns 
numerical_features =X.select_dtypes(exclude='object').columns 
print("Categorical Features:",categorical_features, "\n")
print("Numerical Features:",numerical_features) 

Categorical Features: Index(['Geography', 'Gender'], dtype='str') 

Numerical Features: Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary'],
      dtype='str')


In [21]:
# 1. Split the data FIRST because it prevent from the data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((8000, 10), (2000, 10), (8000,), (2000,))

In [23]:
## Now applying column Transformer  
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder 


Ordinal_encoder_gender = OrdinalEncoder() 
onehot_encoder_geo = OneHotEncoder(sparse_output=False) 
scaler = StandardScaler()

## creating a pipeline
preprocessor = ColumnTransformer(
    transformers=[

        ("OrdinalEncoder",Ordinal_encoder_gender,["Gender"]),
        ("OneHotEncoder",onehot_encoder_geo,["Geography"]), 
        
        #scaling is done only on continuos numerical values not categorical or encoded values like 0 and 1
        ("StandardScaler",scaler,['CreditScore',
            'Age',
            'Tenure',
            'Balance',
            'NumOfProducts',
            'EstimatedSalary'])
    ], 
    remainder="passthrough", ## importatnt because it doesnot drop other remaing num features
    verbose_feature_names_out=False ## features k aage se remainder word hatane k liye dataframe me convert krne k baad
)

#### Important: HasCrCard aur IsActiveMember binary columns hain, isliye unhe scaling se exclude kiya hai. remainder='passthrough' unhe unchanged rakhega.

In [24]:
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

In [25]:
## convert into the dataframe
columns = preprocessor.get_feature_names_out()

X_train = pd.DataFrame(X_train, columns=columns)

X_test = pd.DataFrame(X_test, columns=columns)

In [26]:
print("Training Data:")
display(X_train.head())

print("Testing Data:")
display(X_test.head())

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Data:


,Gender,Geography_France,Geography_Germany,Geography_Spain,CreditScore,Age,Tenure,Balance,NumOfProducts,EstimatedSalary,HasCrCard,IsActiveMember
0,1.0,1.0,0.0,0.0,0.356500,-0.655786,0.345680,-1.218471,0.808436,1.367670,1.0,1.0
1,1.0,0.0,1.0,0.0,-0.203898,0.294938,-0.348369,0.696838,0.808436,1.661254,1.0,1.0
2,1.0,0.0,0.0,1.0,-0.961472,-1.416365,-0.695393,0.618629,-0.916688,-0.252807,1.0,0.0
3,0.0,1.0,0.0,0.0,-0.940717,-1.131148,1.386753,0.953212,-0.916688,0.915393,1.0,0.0
4,1.0,1.0,0.0,0.0,-1.397337,1.625953,1.386753,1.057449,-0.916688,-1.059600,0.0,0.0


Testing Data:


,Gender,Geography_France,Geography_Germany,Geography_Spain,CreditScore,Age,Tenure,Balance,NumOfProducts,EstimatedSalary,HasCrCard,IsActiveMember
0,1.0,0.0,1.0,0.0,-0.577496,-0.655786,-0.695393,0.329937,0.808436,-1.019605,0.0,0.0
1,1.0,1.0,0.0,0.0,-0.297297,0.390011,-1.389442,-1.218471,0.808436,0.798883,1.0,1.0
2,0.0,0.0,0.0,1.0,-0.525607,0.485083,-0.348369,-1.218471,0.808436,-0.727980,1.0,0.0
3,1.0,0.0,1.0,0.0,-1.511492,1.911170,1.039728,0.689272,0.808436,1.221387,1.0,1.0
4,0.0,0.0,0.0,1.0,-0.951094,-1.131148,0.692704,0.782839,-0.916688,0.247560,1.0,1.0


Training Shape: (8000, 12)
Testing Shape: (2000, 12)


### saving all these scaled and encoded (preprocessor) in the pickel file \
* so that if new data or new customer will there we can apply same scalling and encoding techinique without doing again manually fit 

In [27]:
## saving into pickle
import pickle 
with open('preprocessor.pkl',"wb") as file: 
    pickle.dump(preprocessor,file)

## Finally Building ANN MODEL and train

In [28]:
## import Tensorflow libraries  
import tensorflow as tf 
from tensorflow.keras.models import Sequential  #ANN Is sequential means interconnected 
from tensorflow.keras.layers import Dense        # Dense means how much neurone will use in 1 Hidden layers 
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard 


 * Sequential model ka matlab hai data ek layer se next layer mein sequence mein pass hoga.

* Dense layer ka matlab hai ek layer ka har neuron previous layer ke har output se connected hota hai. 

* TensorBoard is used for monitoring the training visualization

In [29]:
## Build our Ann model 
model = Sequential( [
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),   ##HL1 Connected with input Layer 
    Dense(32,activation='relu'),    #Hl2 created 
    Dense(1,activation='sigmoid')   #output layer
]
)

In [30]:
## how many numbers of paramerter are there for training 
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


* basically param means no. of input * weight bias among all the neurones

In [31]:
## before compiling the model best practice is you should define your own requirement  
opt = tf.keras.optimizers.Adam(learning_rate=0.01) 
loss =tf.keras.losses.BinaryCrossentropy() 

In [32]:
## compile model =  forward and backward propagation 
model.compile( 
    optimizer =opt,
    loss = loss, 
    metrics = ['accuracy']
)

* adam: weights update karta hai.
* binary_crossentropy: binary classification ka loss function.
* accuracy: prediction performance measure karta hai.

In [33]:
## set up early stopping  
from tensorflow.keras.callbacks import EarlyStopping 

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# EarlyStopping: Training ko unnecessary continue hone se rokta hai.

* Maan lo validation loss improve nahi ho raha.

* patience=5 ka matlab hai 5 consecutive epochs tak improvement nahi hua toh training stop kar do.

* restore_best_weights=True best validation loss wale epoch ke weights wapas load karega.

In [34]:
# create the path for monitoring the activivty  
import os 
import datetime 

log_dir = os.path.join("logs","fit",datetime.datetime.now().strftime("%Y%m%d-%H%M%S")) 

## set up Tensorboard for training visualization and also configure callback  
from tensorflow.keras.callbacks import TensorBoard

tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

* Ye training ke logs save karega. Baad mein TensorBoard par loss aur accuracy graphs dekh sakte ho.

In [35]:
## Finally the model Training  
history = model.fit( 
    X_train,
    y_train, 
    validation_split= 0.2,
    epochs =100, 
    callbacks = [tensorboard_callback,early_stopping]
)

Epoch 1/100


200/200 [==============================] - 2s 3ms/step - loss: 0.3967 - accuracy: 0.8284 - val_loss: 0.3641 - val_accuracy: 0.8388
Epoch 2/100
200/200 [==============================] - 0s 2ms/step - loss: 0.3580 - accuracy: 0.8500 - val_loss: 0.3547 - val_accuracy: 0.8444
Epoch 3/100
200/200 [==============================] - 0s 2ms/step - loss: 0.3468 - accuracy: 0.8566 - val_loss: 0.3573 - val_accuracy: 0.8569
Epoch 4/100
200/200 [==============================] - 0s 2ms/step - loss: 0.3424 - accuracy: 0.8600 - val_loss: 0.3449 - val_accuracy: 0.8525
Epoch 5/100
200/200 [==============================] - 0s 2ms/step - loss: 0.3395 - accuracy: 0.8619 - val_loss: 0.3478 - val_accuracy: 0.8581
Epoch 6/100
200/200 [==============================] - 0s 2ms/step - loss: 0.3354 - accuracy: 0.8642 - val_loss: 0.3489 - val_accuracy: 0.8519
Epoch 7/100
200/200 [==============================] - 0s 2ms/step - loss: 0.3324 - accuracy: 0.8630 - val_loss: 0.3468 - val_accuracy: 0.85

In [36]:
model.save('model.h5')

In [37]:
model.evaluate(X_test, y_test)

63/63 [==============================] - 0s 1ms/step - loss: 0.3317 - accuracy: 0.8665


[0.3316662609577179, 0.8665000200271606]

In [38]:
## LOad tensorboard Extension 
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [51]:
%tensorboard --logdir logs/fit/20261106-183330

Reusing TensorBoard on port 6007 (pid 28556), started 0:01:00 ago. (Use '!kill 28556' to kill it.)